# 05 — Grover SAT Capstone

The capstone: **Grover's algorithm** generalizes notebook 04's one-query oracle trick to
*search*. An oracle **phase-flips** the amplitude of the state(s) that satisfy your problem;
**diffusion** then reflects every amplitude about the mean, amplifying the marked one. Repeat
~π/4·√N times and the answer dominates — a quadratic speedup over checking N possibilities
one by one.

Here, the "problem" is a small **boolean satisfiability (SAT) formula**. You'll ask Bob —
reaching the Qiskit Code Assistant MCP, whether it's running locally or on a shared remote
server — to author the `PhaseOracle` that marks its satisfying assignment, run Grover on the
Aer simulator, and verify the winner against a classical brute-force search.

Maps to the `/demo` deck slides "The oracle & search" / "Build a SAT oracle with Bob"
(`AmplitudeAmplificationAnimation`).

> **Running this notebook** — from the repo root:
> ```bash
> uv run --with jupyterlab --with qiskit==2.0.0 --with qiskit-algorithms==0.4.0 \
>   --with matplotlib --with pylatexenc --with httpx \
>   jupyter lab notebooks/05-grover-sat-capstone.ipynb
> ```
> This notebook needs a **newer Qiskit than notebooks 01–04** — `PhaseOracle` only drops its
> `tweedledum` dependency from Qiskit 2.0 onward.
>
> Start these in separate terminals for the live steps (both degrade gracefully otherwise):
> ```bash
> cd mcp-server && uv run server.py --http           # IBM Bob's tool, direct HTTP access
> cd backend && uv run uvicorn main:app --reload     # FastAPI circuit execution (needs
>                                                     # qiskit-algorithms — see backend/pyproject.toml)
> ```

## Step 0 — Install dependencies

In [ ]:
%pip install -q qiskit==2.0.0 qiskit-algorithms==0.4.0 matplotlib pylatexenc httpx
print("✅ Qiskit + qiskit-algorithms + matplotlib + pylatexenc + httpx ready")

## Step 0b — Load the workshop helpers (`notebooks/lib/`)

In [ ]:
import sys
from pathlib import Path

_notebooks_dir = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd() / "notebooks"
if str(_notebooks_dir) not in sys.path:
    sys.path.insert(0, str(_notebooks_dir))

from lib.visuals import draw_circuit, plot_counts, run_and_plot
from lib.backend import run_via_backend, explain_with_bob

## Step 1 — Pick a formula

`FORMULA` uses the same `&` / `|` / `~` syntax Qiskit's `PhaseOracle` expects. This one has a
single satisfying assignment among its 8 possible inputs — matching the `/demo` amplitude
animation's 8-state bar chart.

In [ ]:
from itertools import product

VARS = ["a", "b", "c"]
FORMULA = "(a & b) & ~c"


def evaluate(formula: str, assignment: dict[str, int]) -> bool:
    return bool(eval(formula, {"__builtins__": {}}, assignment))


satisfying = [
    "".join(str(bit) for bit in bits)
    for bits in product([0, 1], repeat=len(VARS))
    if evaluate(FORMULA, dict(zip(VARS, bits)))
]
print(f"Formula: {FORMULA}")
print(f"Satisfying assignment(s), in (a, b, c) order: {satisfying}")

## Step 2 — Ask Bob to author the oracle

In IBM Bob's chat, try asking:

> "Build a Qiskit Grover search circuit that finds the satisfying assignment of the boolean
> formula (a & b) & ~c using PhaseOracle."

The cell below calls the same MCP `explain_circuit` tool directly, so you can see what Bob sees
(skips gracefully if the MCP server isn't running in `--http` mode).

In [ ]:
oracle_code = """from qiskit.circuit.library import PhaseOracle

oracle = PhaseOracle("(a & b) & ~c")
"""

explanation = explain_with_bob(oracle_code)
if explanation:
    print(explanation)
else:
    print(
        "MCP server not reachable at http://localhost:8765 \u2014 start it with "
        "`cd mcp-server && uv run server.py --http`, or ask Bob directly in "
        "your assistant chat. Continuing with the oracle below either way."
    )

## Step 3 — Build the Grover circuit

In [ ]:
import math
from qiskit.circuit.library import PhaseOracle
from qiskit_algorithms import Grover, AmplificationProblem

oracle = PhaseOracle(FORMULA)
problem = AmplificationProblem(oracle)

n_iterations = round(math.pi / 4 * math.sqrt(2**oracle.num_qubits))
print(f"Qubits: {oracle.num_qubits}, optimal iterations \u2248 \u03c0/4\u00b7\u221aN \u2248 {n_iterations}")

grover = Grover(iterations=n_iterations)
qc = grover.construct_circuit(problem, power=n_iterations, measurement=True)

draw_circuit(qc, title=f"Grover, {n_iterations} iteration(s)")

## Step 4 — Run it (FastAPI + Aer)

Same run path as notebooks 02–03 — `/api/circuit/run` — just with a bigger circuit.

In [ ]:
grover_code = f'''from qiskit.circuit.library import PhaseOracle
from qiskit_algorithms import Grover, AmplificationProblem

oracle = PhaseOracle({FORMULA!r})
problem = AmplificationProblem(oracle)
grover = Grover(iterations={n_iterations})
qc = grover.construct_circuit(problem, power={n_iterations}, measurement=True)
'''

counts = run_via_backend(grover_code, timeout=60.0)

if counts is None:
    print("Backend not reachable at http://localhost:8000 \u2014 running locally instead.")
    counts = run_and_plot(qc, title="Grover result (local fallback)")
else:
    print("Ran via FastAPI backend (Aer simulator).")
    plot_counts(counts, title="Grover result (via /api/circuit/run)")

## Step 5 — Read it & verify

In [ ]:
winner = max(counts, key=counts.get)
# Qiskit's classical register is little-endian and PhaseOracle assigns qubits in the order
# variables first appear (a, b, c) — reverse the bitstring to read it back in (a, b, c) order.
assignment_str = winner[::-1]
pct = counts[winner] / sum(counts.values()) * 100

print(
    f"Most frequent outcome: {winner} \u2192 a={assignment_str[0]}, b={assignment_str[1]}, "
    f"c={assignment_str[2]} ({pct:.1f}% of shots)"
)
print(f"Brute-force satisfying assignment(s): {satisfying}")
print(f"Match: {assignment_str in satisfying}")
assert assignment_str in satisfying

## What's next?

`PhaseOracle` + `Grover` gives a **provable** quadratic speedup for SAT — the oracle is
constructed directly from the formula, no quantum arithmetic required. For *optimization*
problems (QUBO), the same amplitude-amplification idea extends via Grover Adaptive Search —
ask Bob to help build a QUBO oracle as a forward-looking next step beyond this workshop.

---
*Quantum Experience Lab — IBM Bob + Qiskit workshop*